# Solutions - Notebook 07 exercises

Try each task yourself first. Every solution is a few lines and prints its answer.

In [ ]:
import sys                                    # lets us change where Python looks for files
sys.path.append("../..")                      # utils.py and matcher.py live two folders up
import cv2                                    # OpenCV
import numpy as np                            # NumPy
from utils import load       # our helpers
print("Libraries loaded - ready to go!")      # a visible sign that everything worked

### Task 1 - In `objects_shapes.png`, print the area of the biggest green square.

In [ ]:
img = load("../../assets/objects_shapes.png")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, np.array([40, 100, 100]), np.array([80, 255, 255]))   # green
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
biggest = 0
for contour in contours:
    if cv2.contourArea(contour) > biggest:
        biggest = cv2.contourArea(contour)
print("green objects:", len(contours), "| biggest area:", biggest, "(80 x 80 = 6400, minus edges)")

### Task 2 - In `objects_scene.png`, count the circles with an area above 5000.

In [ ]:
scene = load("../../assets/objects_scene.png")
gray = cv2.GaussianBlur(cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY), (5, 5), 0)
_, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
contours, _ = cv2.findContours(cv2.bitwise_not(otsu), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
big_circles = 0
for contour in contours:
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    if area > 5000 and 4 * np.pi * area / (perimeter * perimeter) >= 0.85:
        big_circles = big_circles + 1
print("big circles:", big_circles)

### Task 3 - Cut the big square plus a 10-pixel white margin (rows 220-330, columns 240-350) out of the scene and find it again with `cv2.matchTemplate`. (Without the margin the template is one flat colour, and a flat template cannot be matched.)

In [ ]:
gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
template = gray[220:331, 240:351]                          # the square plus a white margin
result = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
print("best score:", round(float(max_val), 2), "at (x, y) =", max_loc)